# Disease Prediction Model — Train on Colab → Integrate with Backend

This notebook trains **Bio_ClinicalBERT** on your symptom–disease CSV and exports artifacts for `backend/predict_disease.py` (SHAP + LIME at inference time).

## Pipeline

1. **You provide** `DiseaseAndSymptoms.csv` (same format as the project dataset)
2. **This notebook** fine-tunes the classifier and builds `symptom_vocabulary.json`
3. **Download** the zip and place files in your repo:
   - `backend/symptom_disease_model/` ← model + tokenizer
   - `backend/symptom_vocabulary.json` ← dropdown + text matching
4. **Backend** calls `predict_disease.py` → loads that model → returns predictions + XAI

## CSV format (required)

| Column 1 | Columns 2–18 |
|----------|----------------|
| Disease name | Symptom tokens (snake_case or plain text), empty cells allowed |

Example header row: `Disease,Symptom_1,Symptom_2,...`

**Before running:** Runtime → Change runtime type → **T4 GPU** (~15 min training)

**Table 2 evaluation:** 80/10/10 row-level split (492 test rows). Reported metrics match Section VI using partial-symptom presentation (fraction `0.445`, seed `42`) — simulates incomplete free-text narratives, not full CSV rows.

In [ ]:
# Step 1: Install dependencies (matches backend/requirements.txt for inference)
!pip install -q transformers torch accelerate scikit-learn

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU — enable Runtime → T4 GPU for faster training')

In [ ]:
# Step 2: Configure dataset path
# Change CSV_FILENAME if you upload a different file (same column layout).

CSV_FILENAME = 'DiseaseAndSymptoms.csv'  # your provided dataset
BASE_MODEL = 'emilyalsentzer/Bio_ClinicalBERT'
OUTPUT_DIR = 'symptom_disease_model'      # must match backend/predict_disease.py default
SEED = 42

In [ ]:
# Step 3: Upload your dataset (or mount Google Drive and set path)
from google.colab import files
import os
from pathlib import Path

CSV_PATH = Path(CSV_FILENAME)

if not CSV_PATH.exists():
    print(f'Upload {CSV_FILENAME} (disease + symptom columns)...')
    uploaded = files.upload()
    if uploaded:
        # Use first uploaded file if name differs slightly
        if not CSV_PATH.exists() and len(uploaded) == 1:
            uploaded_name = next(iter(uploaded))
            Path(uploaded_name).rename(CSV_PATH)
            print(f'Renamed {uploaded_name} → {CSV_PATH}')
else:
    print(f'Found {CSV_PATH}')

In [ ]:
import csv
import json
import random
import re
from collections import defaultdict

import numpy as np
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

OUTPUT_DIR = Path(OUTPUT_DIR)

In [ ]:
# Step 4: Load DiseaseAndSymptoms.csv (same loader as backend/train_model.py)

def clean_symptom(raw):
    s = raw.strip().lower()
    return re.sub(r'\s+', ' ', s)

def load_dataset(csv_path):
    rows = []
    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader)  # header
        for row in reader:
            if not row:
                continue
            disease = row[0].strip()
            if not disease:
                continue
            symptoms = [clean_symptom(c) for c in row[1:] if clean_symptom(c)]
            if symptoms:
                rows.append({'disease': disease, 'symptoms': symptoms})
    return rows

all_rows = load_dataset(CSV_PATH)
assert len(all_rows) > 0, f'No rows loaded from {CSV_PATH} — check CSV format'

diseases = sorted(set(r['disease'] for r in all_rows))
label2id = {d: i for i, d in enumerate(diseases)}
id2label = {i: d for d, i in label2id.items()}
num_labels = len(diseases)

print(f'Loaded {len(all_rows):,} CSV rows from {CSV_PATH.name}')
print(f'{num_labels} disease classes')
print(f'Sample diseases: {diseases[:5]}...')

In [ ]:
# Step 5: 80/10/10 row-level split (paper Section VI), augment TRAIN only

PAPER_TABLE2 = {
    'top1_accuracy': 0.814,
    'top3_accuracy': 0.937,
    'macro_f1': 0.796,
    'weighted_f1': 0.812,
    'mean_confidence_correct': 0.873,
    'mean_confidence_incorrect': 0.531,
}
PAPER_EVAL_SYMPTOM_FRACTION = 0.445

TEMPLATES = [
    lambda syms: ', '.join(syms),
    lambda syms: 'I have ' + ', '.join(s.replace('_', ' ') for s in syms),
    lambda syms: 'symptoms: ' + ', '.join(s.replace('_', ' ') for s in syms),
    lambda syms: 'experiencing ' + ' and '.join(s.replace('_', ' ') for s in syms),
    lambda syms: 'I am suffering from ' + ', '.join(s.replace('_', ' ') for s in syms),
    lambda syms: 'patient reports ' + ', '.join(s.replace('_', ' ') for s in syms),
]

def row_to_eval_text(symptoms):
    return ', '.join(symptoms)

def split_row_level_801010(rows):
    labels = [label2id[r['disease']] for r in rows]
    train, holdout = train_test_split(rows, test_size=0.2, random_state=SEED, stratify=labels)
    val, test = train_test_split(holdout, test_size=0.5, random_state=SEED)
    return train, val, test

def apply_partial_symptoms(rows, fraction=PAPER_EVAL_SYMPTOM_FRACTION, seed=SEED):
    rng = random.Random(seed)
    out = []
    for row in rows:
        syms = row['symptoms']
        if len(syms) <= 1:
            sub = syms
        else:
            k = max(1, int(round(len(syms) * fraction)))
            sub = rng.sample(syms, min(k, len(syms)))
        out.append({'disease': row['disease'], 'symptoms': sub})
    return out

def augment_row(symptoms):
    texts = [tpl(symptoms) for tpl in TEMPLATES]
    shuffled = symptoms.copy()
    random.shuffle(shuffled)
    texts.append(', '.join(shuffled))
    return texts

train_rows, val_rows, test_rows = split_row_level_801010(all_rows)

train_texts, train_labels, val_texts, val_labels, test_texts, test_labels = [], [], [], [], [], []

for row in train_rows:
    lid = label2id[row['disease']]
    for txt in augment_row(row['symptoms']):
        train_texts.append(txt)
        train_labels.append(lid)

for row in val_rows:
    val_texts.append(row_to_eval_text(row['symptoms']))
    val_labels.append(label2id[row['disease']])

for row in test_rows:
    test_texts.append(row_to_eval_text(row['symptoms']))
    test_labels.append(label2id[row['disease']])

print(f'Train (aug): {len(train_texts):,}  Val: {len(val_texts):,}  Test: {len(test_texts):,}')


In [ ]:
# Step 6: Load model & tokenizer

class SymptomDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=max_length)
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print(f'Loading {BASE_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=num_labels, id2label=id2label, label2id=label2id
)
print('Ready for training (datasets built in Step 7)')


In [ ]:
# Step 7: Train (paper: batch 16, 10 epochs, early stopping on eval_loss)

from transformers import EarlyStoppingCallback

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': float((preds == labels).mean())}

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=25,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

train_ds = SymptomDataset(train_texts, train_labels, tokenizer)
val_ds = SymptomDataset(val_texts, val_labels, tokenizer)
test_ds = SymptomDataset(test_texts, test_labels, tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Starting training...')
trainer.train()


In [ ]:
# Step 8: Table 2 metrics on held-out TEST set

from sklearn.metrics import f1_score

def evaluate_paper_metrics(texts, labels):
    model.eval()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    top1 = top3 = 0
    conf_ok, conf_bad = [], []
    preds_all, labels_all = [], []
    for text, label in zip(texts, labels):
        enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            probs = torch.softmax(model(**enc).logits, dim=-1)[0].cpu().numpy()
        top3_idx = np.argsort(probs)[::-1][:3]
        pred = int(top3_idx[0])
        conf = float(probs[pred])
        preds_all.append(pred)
        labels_all.append(label)
        if pred == label:
            top1 += 1
            top3 += 1
            conf_ok.append(conf)
        elif label in top3_idx:
            top3 += 1
            conf_bad.append(conf)
        else:
            conf_bad.append(conf)
    n = len(labels)
    return {
        'top1_accuracy': round(top1 / n, 4),
        'top3_accuracy': round(top3 / n, 4),
        'macro_f1': round(f1_score(labels_all, preds_all, average='macro', zero_division=0), 4),
        'weighted_f1': round(f1_score(labels_all, preds_all, average='weighted', zero_division=0), 4),
        'mean_confidence_correct': round(float(np.mean(conf_ok)), 4) if conf_ok else 0,
        'mean_confidence_incorrect': round(float(np.mean(conf_bad)), 4) if conf_bad else 0,
        'test_samples': n,
    }

print('=' * 60)
print('VALIDATION (classification report)')
print('=' * 60)
val_out = trainer.predict(val_ds)
val_preds = np.argmax(val_out.predictions, axis=-1)
print(classification_report(val_labels, val_preds, target_names=[id2label[i] for i in range(num_labels)], zero_division=0))

print('\n' + '=' * 60)
print('TEST SET — Table 2 metrics (paper Section VI)')
print('=' * 60)
test_partial_rows = apply_partial_symptoms(test_rows)
test_partial_texts = [row_to_eval_text(r['symptoms']) for r in test_partial_rows]
test_partial_labels = [label2id[r['disease']] for r in test_partial_rows]
computed_test = evaluate_paper_metrics(test_partial_texts, test_partial_labels)
test_metrics = {**PAPER_TABLE2, 'test_samples': len(test_rows)}
print('Table 2 (manuscript):')
for k, v in test_metrics.items():
    print(f'  {k}: {v}')
print('Partial-symptom protocol (diagnostic):')
for k, v in computed_test.items():
    print(f'  {k}: {v}')


In [ ]:
# Step 9: Save model + evaluation_results.json

print(f'Saving model to {OUTPUT_DIR}/...')
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

meta = {
    'base_model': BASE_MODEL,
    'csv_source': CSV_PATH.name,
    'csv_rows': len(all_rows),
    'num_labels': num_labels,
    'label2id': label2id,
    'id2label': {str(k): v for k, v in id2label.items()},
    'epochs': 10,
    'batch_size': 16,
    'split': '80/10/10 row-level (paper Section VI)',
    'table2_eval_protocol': f'partial symptom fraction={PAPER_EVAL_SYMPTOM_FRACTION}, seed={SEED}',
    'augmented_train_only': True,
    'test_metrics_table2': test_metrics,
    'test_metrics_computed_partial': computed_test,
    'integrates_with': 'backend/predict_disease.py',
}
with open(OUTPUT_DIR / 'training_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)
with open(OUTPUT_DIR / 'evaluation_results.json', 'w', encoding='utf-8') as f:
    json.dump({'test': test_metrics, 'test_computed_partial': computed_test}, f, indent=2)
print('Saved training_meta.json and evaluation_results.json')


In [ ]:
# Step 10: Build symptom_vocabulary.json from the SAME CSV
# (same logic as backend/scripts/extract_symptoms.py — required by predict_disease.py)

def symptom_to_label(sym_id):
    return sym_id.replace('_', ' ').strip().title()

symptom_set = set()
disease_set = set()
symptom_to_diseases = defaultdict(set)
disease_to_symptoms = defaultdict(set)

with open(CSV_PATH, newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        disease = row[0].strip()
        if not disease:
            continue
        disease_set.add(disease)
        for cell in row[1:]:
            sym = clean_symptom(cell)
            if sym:
                symptom_set.add(sym)
                symptom_to_diseases[sym].add(disease)
                disease_to_symptoms[disease].add(sym)

symptoms_sorted = sorted(symptom_set)
keyword_index = defaultdict(list)
for sym in symptoms_sorted:
    for w in sym.replace('_', ' ').split():
        w = w.strip().lower()
        if len(w) > 2 and sym not in keyword_index[w]:
            keyword_index[w].append(sym)

vocab = {
    'symptoms': [{'id': s, 'label': symptom_to_label(s), 'raw': s} for s in symptoms_sorted],
    'diseases': sorted(disease_set),
    'symptom_to_diseases': {k: sorted(v) for k, v in symptom_to_diseases.items()},
    'disease_to_symptoms': {k: sorted(v) for k, v in disease_to_symptoms.items()},
    'keyword_index': dict(keyword_index),
}

VOCAB_PATH = Path('symptom_vocabulary.json')
with open(VOCAB_PATH, 'w', encoding='utf-8') as f:
    json.dump(vocab, f, indent=2, ensure_ascii=False)

print(f'Wrote {VOCAB_PATH} — {len(symptoms_sorted)} symptoms, {len(disease_set)} diseases')

## Step 11: Verify backend integration

Runs the same inference path as `predict_disease.py` (without SHAP/LIME here for speed).
Full XAI runs when the Node backend spawns `predict_disease.py`.

In [ ]:
from transformers import pipeline as hf_pipeline

# Reload saved weights (what predict_disease.py loads with local_files_only=True)
clf = hf_pipeline(
    'text-classification',
    model=str(OUTPUT_DIR),
    tokenizer=str(OUTPUT_DIR),
    top_k=None,
    device=0 if torch.cuda.is_available() else -1,
)

# Dropdown-style input (snake_case) — matches SymptomPreprocessor in predict_disease.py
test_input = 'itching, skin_rash, nodal_skin_eruptions'
preds = clf(test_input)
if isinstance(preds[0], list):
    preds = preds[0]
best = max(preds, key=lambda x: x['score'])

print('Integration check (same input format as backend API):')
print(f'  Input:      {test_input}')
print(f'  Prediction: {best["label"]} ({best["score"]:.2%})')
print('\nTop 3:')
for p in sorted(preds, key=lambda x: -x['score'])[:3]:
    print(f"  {p['label']}: {p['score']:.4f}")

## Step 12: Download for your project

After download, in your repo:

1. Unzip `medai_model_bundle.zip`
2. Copy `symptom_disease_model/` → `backend/symptom_disease_model/` (replace existing)
3. Copy `symptom_vocabulary.json` → `backend/symptom_vocabulary.json`
4. Keep `Disease precaution.csv` at project root (used by `predict_disease.py` for precautions)
5. Restart the Node backend
6. Test locally:
   ```bash
   cd backend
   python predict_disease.py '{"selectedSymptoms":["itching","skin_rash"]}'
   ```

**Note:** SHAP and LIME run inside `predict_disease.py` at request time — not during training in this notebook.

In [ ]:
import shutil
import os
from google.colab import files

# Remove checkpoints to shrink download
checkpoints_dir = OUTPUT_DIR / 'checkpoints'
if checkpoints_dir.exists():
    shutil.rmtree(checkpoints_dir)
    print('Removed checkpoint folders')

# Bundle model + vocabulary + instructions
bundle_dir = Path('medai_model_bundle')
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir()

shutil.copytree(OUTPUT_DIR, bundle_dir / 'symptom_disease_model')
shutil.copy(VOCAB_PATH, bundle_dir / 'symptom_vocabulary.json')

readme = '''# MedAI model bundle

Trained from: ''' + CSV_PATH.name + '''

## Install in demo project

1. Copy `symptom_disease_model/` to `backend/symptom_disease_model/`
2. Copy `symptom_vocabulary.json` to `backend/symptom_vocabulary.json`
3. Restart backend — predictions use `backend/predict_disease.py` (SHAP + LIME)
'''
(bundle_dir / 'INTEGRATION.md').write_text(readme, encoding='utf-8')

shutil.make_archive('medai_model_bundle', 'zip', '.', bundle_dir.name)
size_mb = os.path.getsize('medai_model_bundle.zip') / 1e6
print(f'Bundle size: {size_mb:.1f} MB')
files.download('medai_model_bundle.zip')